# Phase 16: YOLO11m + P2 Head (Sanity Run)

This notebook dynamically creates the `yolo11-p2.yaml` architecture, loads `yolo11m.pt` weights (verifying transfer), and runs a 3-epoch sanity training to ensure mathematically correct operation on the Tesla T4 without OOM.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.5 MB/s eta 0:00:00


In [3]:
import os
import torch
from ultralytics import YOLO

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Current working directory: /content/drive/MyDrive/sem_defect_project
CUDA Available: True
GPU Device: Tesla T4


In [4]:
# Create Custom yolo11-p2.yaml
yaml_content = '''
# YOLO11-P2 (Stride-4 High Resolution Head)
nc: 1
scales:
  # [depth, width, max_channels]
  m: [0.50, 0.50, 512] # YOLO11m scale

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]] # 2
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]] # 4
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]] # 6
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]] # 8
  - [-1, 1, SPPF, [1024, 5]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, False]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)

  # --- P2 EXTENSION ---
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]] # cat backbone P2
  - [-1, 2, C3k2, [128, False]] # 19 (P2/4-tiny)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P3
  - [-1, 2, C3k2, [256, False]] # 22 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, False]] # 25 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 2, C3k2, [1024, True]] # 28 (P5/32-large)

  - [[19, 22, 25, 28], 1, Detect, [nc]] # Detect(P2, P3, P4, P5)
'''

with open('yolo11-p2.yaml', 'w') as f:
    f.write(yaml_content)

print("Created yolo11-p2.yaml successfully!")


Created yolo11-p2.yaml successfully!


In [5]:
# Instantiate Model and Load Weights
print("Instantiating YOLO11m-P2 model...")
model = YOLO('yolo11-p2.yaml')

print("\nLoading pretrained YOLO11m weights into P2 architecture...")
# Load weights, matching shapes automatically
model.load('yolo11m.pt')

# Print full architecture summary to verify P2/P3/P4/P5 shapes
model.info(detailed=True)


Instantiating YOLO11m-P2 model...
WARNING ⚠️ no model scale passed. Assuming scale='m'.

Loading pretrained YOLO11m weights into P2 architecture...
Transferred 88/803 items from pretrained weights
layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d      True         864       [32, 3, 3, 3] -0.000242     0.106        float32
    1                       model.0.bn.weight         BatchNorm2d      True          32                [32]         1         0        float32
    1                         model.0.bn.bias         BatchNorm2d      True          32                [32]         0         0        float32
    2                             model.0.act                SiLU     False           0                  []         -         -              -
    3                     model.1.conv.weight              Conv2d      True       18432      [64, 32, 3

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/torch_utils.py:555: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  f"{i:>5g}{f'{mn}.{pn}':>40}{mt:>20}{p.requires_grad!r:>10}{p.numel():>12g}{list(p.shape)!s:>20}{p.mean():>10.3g}{p.std():>10.3g}{str(p.dtype).replace('torch.', ''):>15}"


YOLO11-p2 summary: 286 layers, 5,481,460 parameters, 5,481,444 gradients, 26.3 GFLOPs


(286, 5481460, 5481444, 26.2783488)

## 2. Sanity Training Run

In [6]:
import time

print("Starting 3-Epoch Sanity Run (Batch=16, 640x640)...")
t0 = time.time()

results = model.train(
    data='dataset_yolo_single_class/data.yaml',
    epochs=3,
    imgsz=640,
    batch=16,
    name='EXP16-Sanity-YOLO11m-P2',
    cache=False,
    amp=True,
    exist_ok=True
)

t1 = time.time()
print(f"\nSanity Run Complete in {t1 - t0:.2f} seconds.")


Starting 3-Epoch Sanity Run (Batch=16, 640x640)...
Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11-p2.yaml, momentum=0.937, mosaic=1.0, multi_scal

In [7]:
# Print Peak VRAM Usage
print("\n=== TESLA T4 VRAM USAGE ===")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv



=== TESLA T4 VRAM USAGE ===
memory.used [MiB], memory.total [MiB]
637 MiB, 15360 MiB
